<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Arimax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install if needed
!pip install yfinance statsmodels scikit-learn -q

import yfinance as yf
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import numpy as np

# -------------------------
# Download Data
# -------------------------

start = "2015-01-01"

nifty = yf.download("^NSEI", start=start, auto_adjust=True)
vix = yf.download("^INDIAVIX", start=start, auto_adjust=True)
oil = yf.download("CL=F", start=start, auto_adjust=True)
gold = yf.download("GC=F", start=start, auto_adjust=True)
usd = yf.download("INR=X", start=start, auto_adjust=True)

# Flatten MultiIndex columns
for df in [nifty, vix, oil, gold, usd]:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

# -------------------------
# Create Dataset
# -------------------------

data = pd.DataFrame()

data["NIFTY"] = nifty["Close"]
data["VIX"] = vix["Close"]
data["Oil"] = oil["Close"]
data["Gold"] = gold["Close"]
data["USDINR"] = usd["Close"]

data = data.dropna()

# Daily Returns
data["Return"] = data["NIFTY"].pct_change()

data = data.dropna()

# -------------------------
# Train Test Split
# -------------------------

train_size = int(len(data)*0.8)

train = data.iloc[:train_size]
test = data.iloc[train_size:]

# -------------------------
# ARIMA
# -------------------------

arima = SARIMAX(
    train["Return"],
    order=(1,0,1)
)

arima_fit = arima.fit(disp=False)

pred_arima = arima_fit.forecast(len(test))

# -------------------------
# ARIMAX
# -------------------------

exog_train = train[["VIX","Oil","Gold","USDINR"]]
exog_test = test[["VIX","Oil","Gold","USDINR"]]

arimax = SARIMAX(
    train["Return"],
    exog=exog_train,
    order=(1,0,1)
)

arimax_fit = arimax.fit(disp=False)

pred_arimax = arimax_fit.forecast(
    len(test),
    exog=exog_test
)

# -------------------------
# Evaluation
# -------------------------

print("ARIMA")

print("RMSE:",
      np.sqrt(mean_squared_error(
          test["Return"],
          pred_arima
      )))

print("MAE:",
      mean_absolute_error(
          test["Return"],
          pred_arima
      ))

print()

print("ARIMAX")

print("RMSE:",
      np.sqrt(mean_squared_error(
          test["Return"],
          pred_arimax
      )))

print("MAE:",
      mean_absolute_error(
          test["Return"],
          pred_arimax
      ))

# -------------------------
# Which Model Won?
# -------------------------

rmse1 = np.sqrt(mean_squared_error(test["Return"], pred_arima))
rmse2 = np.sqrt(mean_squared_error(test["Return"], pred_arimax))

if rmse2 < rmse1:
    print("\nConclusion:")
    print("Adding macroeconomic variables improved forecasting.")
else:
    print("\nConclusion:")
    print("Macroeconomic variables did not improve forecasting.")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be gi

ARIMA
RMSE: 0.008860931101486108
MAE: 0.006266759280479712

ARIMAX
RMSE: 0.010849181142043668
MAE: 0.008321165767295883

Conclusion:
Macroeconomic variables did not improve forecasting.


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
